In [ ]:
import os, json, random, shutil
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

INPUT_ROOT = Path("/kaggle/input")
WORK       = Path("/kaggle/working")

YOUR_COCO_JSON = "/kaggle/input/datasets/vidishagar/pcb-large-coco/annotations/instances_default.json"   # Path("/kaggle/input/my-pcb/instances_default.json")
YOUR_IMG_ROOT  = "/kaggle/input/datasets/vidishagar/pcb-large-dataset/categories"   # Path("/kaggle/input/my-pcb/categories")
NEG_IMG_ROOT   = "/kaggle/input/datasets/vidishagar/non-def/non-def"   # Path("/kaggle/input/my-negatives")

# the 8 well-populated classes to keep
KEEP = ["Component Liftup","Component Missing","Component No Solder",
        "Component Solder Dry","Polarity Wrong","RYB Wrong Sequence",
        "Solder Ball","Solder Short"]

MODEL       = "yolo11s.pt"
IMG_SIZE    = 640
BATCH       = 16
EPOCHS      = 150
SEED        = 42
VAL_FRAC    = 0.20
N_NEG_TOTAL = 120

IMAGE_EXTS = [".jpg",".jpeg",".png",".bmp",".JPG",".JPEG",".PNG",".BMP"]
random.seed(SEED)

In [4]:
def coco_bbox_to_yolo(b,W,H):
    x,y,w,h=b
    return (x+w/2)/W,(y+h/2)/H,w/W,h/H

def write_yolo_split(coco, ids, split, out_root, src_lookup, names):
    (out_root/f"images/{split}").mkdir(parents=True,exist_ok=True)
    (out_root/f"labels/{split}").mkdir(parents=True,exist_ok=True)
    cats=sorted(coco["categories"],key=lambda c:c["id"])
    cid2yolo={c["id"]:i for i,c in enumerate(cats)}
    byid={im["id"]:im for im in coco["images"]}
    anns=defaultdict(list)
    for a in coco["annotations"]: anns[a["image_id"]].append(a)
    ni=nb=0
    for i in ids:
        im=byid[i]; src=src_lookup(im["file_name"])
        if src is None: continue
        uniq=im["file_name"].replace("/","__").replace("\\","__")
        shutil.copy(src, out_root/f"images/{split}/{uniq}")
        W,H=im["width"],im["height"]; lines=[]
        for a in anns.get(i,[]):
            cx,cy,w,h=coco_bbox_to_yolo(a["bbox"],W,H)
            lines.append(f"{cid2yolo[a['category_id']]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            nb+=1
        (out_root/f"labels/{split}/{Path(uniq).stem}.txt").write_text("\n".join(lines))
        ni+=1
    return ni,nb

def add_negatives(neg_paths, out_root, n_total, val_frac, seed=SEED):
    if not neg_paths: print("no negatives"); return
    negs=neg_paths[:]; random.Random(seed).shuffle(negs); negs=negs[:n_total]
    nval=round(n_total*val_frac)
    for split,chunk in (("val",negs[:nval]),("train",negs[nval:])):
        (out_root/f"images/{split}").mkdir(parents=True,exist_ok=True)
        (out_root/f"labels/{split}").mkdir(parents=True,exist_ok=True)
        for p in chunk:
            shutil.copy(p, out_root/f"images/{split}/{p.name}")
            (out_root/f"labels/{split}/{p.stem}.txt").write_text("")
    print(f"negatives: {len(negs)-nval} train, {nval} val")

def filter_coco_to_classes(coco, keep_names):
    keep_set=set(keep_names)
    id2name={c["id"]:c["name"] for c in coco["categories"]}
    new_cats=[{"id":i+1,"name":n} for i,n in enumerate(keep_names)]
    name2new={c["name"]:c["id"] for c in new_cats}
    new_anns=[]
    for a in coco["annotations"]:
        nm=id2name.get(a["category_id"])
        if nm in keep_set:
            b=dict(a); b["category_id"]=name2new[nm]; new_anns.append(b)
    imgs_with_ann={a["image_id"] for a in new_anns}
    new_imgs=[im for im in coco["images"] if im["id"] in imgs_with_ann]
    return {"images":new_imgs,"annotations":new_anns,"categories":new_cats}

In [5]:
if YOUR_COCO_JSON is None:
    for c in INPUT_ROOT.rglob("*.json"):
        try:
            d=json.loads(Path(c).read_text())
            if "annotations" in d and "categories" in d:
                YOUR_COCO_JSON=c; break
        except Exception: pass
print("YOUR_COCO_JSON =", YOUR_COCO_JSON)
assert YOUR_COCO_JSON, "Set YOUR_COCO_JSON in CELL 0."

your_coco=json.loads(Path(YOUR_COCO_JSON).read_text())
YOUR_NAMES=[c["name"] for c in sorted(your_coco["categories"],key=lambda c:c["id"])]
print(f"{len(YOUR_NAMES)} classes in file:",YOUR_NAMES)

if YOUR_IMG_ROOT is None: YOUR_IMG_ROOT=Path(YOUR_COCO_JSON).parent
IMG_BASE=Path(YOUR_IMG_ROOT).parent
your_src={}
for p in Path(YOUR_IMG_ROOT).rglob("*"):
    if p.suffix.lower() in (".jpg",".jpeg",".png",".bmp"):
        your_src[str(p.relative_to(IMG_BASE))]=p
print("your images found:",len(your_src))

_bidx=defaultdict(list)
for k,v in your_src.items(): _bidx[Path(k).name].append(v)
def resolve_your_image(fn):
    if fn in your_src: return your_src[fn]
    h=_bidx.get(Path(fn).name,[]); return h[0] if len(h)==1 else None

found=sum(1 for im in your_coco["images"] if resolve_your_image(im["file_name"]))
print(f"COCO resolvable: {found}/{len(your_coco['images'])}")
assert found>=len(your_coco["images"])*0.98

if NEG_IMG_ROOT is None:
    for r in (INPUT_ROOT.iterdir() if INPUT_ROOT.exists() else []):
        if any(k in r.name.lower() for k in ("neg","good","clean","ok","defect-free")):
            NEG_IMG_ROOT=r; break
neg_files=[p for p in Path(NEG_IMG_ROOT).rglob("*")
           if p.suffix.lower() in (".jpg",".jpeg",".png",".bmp")] if NEG_IMG_ROOT else []
print("negatives:",len(neg_files),"from",NEG_IMG_ROOT)

YOUR_COCO_JSON = /kaggle/input/datasets/vidishagar/pcb-large-coco/annotations/instances_default.json
12 classes in file: ['Component Crack', 'Component Damage', 'Component Liftup', 'Component Missing', 'Component No Solder', 'Component Solder Dry', 'LED Damage', 'Polarity Wrong', 'RYB Wrong Sequence', 'Solder Ball', 'Solder Short', 'Tombstone']
your images found: 551
COCO resolvable: 548/551
negatives: 150 from /kaggle/input/datasets/vidishagar/non-def/non-def


In [6]:
missing = [k for k in KEEP if k not in YOUR_NAMES]
assert not missing, f"These KEEP names aren't in YOUR_NAMES: {missing}"

coco8 = filter_coco_to_classes(your_coco, KEEP)
NAMES8 = [c["name"] for c in sorted(coco8["categories"],key=lambda c:c["id"])]
id2n = {c["id"]:c["name"] for c in coco8["categories"]}
print(f"8-class dataset: {len(coco8['images'])} images, {len(coco8['annotations'])} annotations")
cc=Counter(a["category_id"] for a in coco8["annotations"])
for cid in sorted(cc): print(f"  {id2n[cid]:<22} {cc[cid]}")

8-class dataset: 508 images, 565 annotations
  Component Liftup       22
  Component Missing      44
  Component No Solder    114
  Component Solder Dry   138
  Polarity Wrong         36
  RYB Wrong Sequence     24
  Solder Ball            43
  Solder Short           144


In [11]:
cls_freq=Counter(a["category_id"] for a in coco8["annotations"])
img_cls=defaultdict(set)
for a in coco8["annotations"]: img_cls[a["image_id"]].add(a["category_id"])
strat=defaultdict(list)
for im in coco8["images"]:
    cls=img_cls.get(im["id"],set())
    key=min(cls,key=lambda c:cls_freq[c]) if cls else "__none__"
    strat[key].append(im["id"])
rng=random.Random(SEED); y_train=set(); y_val=set()
for k,ids in strat.items():
    ids=ids[:]; rng.shuffle(ids)
    nv=max(1,round(len(ids)*VAL_FRAC)) if len(ids)>1 else 0
    y_val|=set(ids[:nv]); y_train|=set(ids[nv:])
assert not (y_train & y_val)
print(f"train {len(y_train)} | val {len(y_val)}")
for c in sorted(cls_freq):
    ntr=sum(1 for a in coco8["annotations"] if a["category_id"]==c and a["image_id"] in y_train)
    nva=sum(1 for a in coco8["annotations"] if a["category_id"]==c and a["image_id"] in y_val)
    print(f"  {id2n[c]:<22} train {ntr:>3} val {nva:>3}")

train 407 | val 101
  Component Liftup       train  18 val   4
  Component Missing      train  35 val   9
  Component No Solder    train  97 val  17
  Component Solder Dry   train 111 val  27
  Polarity Wrong         train  29 val   7
  RYB Wrong Sequence     train  19 val   5
  Solder Ball            train  35 val   8
  Solder Short           train 114 val  30


In [12]:
import yaml
YOUR_YOLO=WORK/"yolo8"; shutil.rmtree(YOUR_YOLO, ignore_errors=True)
ysl=lambda n: resolve_your_image(n)
print("pos train:",write_yolo_split(coco8,y_train,"train",YOUR_YOLO,ysl,NAMES8))
print("pos val  :",write_yolo_split(coco8,y_val,  "val",  YOUR_YOLO,ysl,NAMES8))
add_negatives(neg_files,YOUR_YOLO,N_NEG_TOTAL,VAL_FRAC)
(YOUR_YOLO/"data.yaml").write_text(yaml.safe_dump(
    {"path":str(YOUR_YOLO),"train":"images/train","val":"images/val",
     "names":{i:n for i,n in enumerate(NAMES8)}},sort_keys=False))
print("data.yaml written; classes:",NAMES8)

pos train: (404, 453)
pos val  : (101, 107)
negatives: 96 train, 24 val
data.yaml written; classes: ['Component Liftup', 'Component Missing', 'Component No Solder', 'Component Solder Dry', 'Polarity Wrong', 'RYB Wrong Sequence', 'Solder Ball', 'Solder Short']


In [13]:
YR=YOUR_YOLO
for split in ["train","val"]:
    imgs=[p for p in (YR/f"images/{split}").rglob("*") if p.suffix.lower() in (".jpg",".jpeg",".png",".bmp")]
    labs=list((YR/f"labels/{split}").rglob("*.txt"))
    nonempty=[p for p in labs if p.read_text().strip()]
    print(f"[{split}] images={len(imgs)} labels={len(labs)} non-empty={len(nonempty)}")

[train] images=500 labels=500 non-empty=404
[val] images=125 labels=125 non-empty=101


In [15]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.4 MB/s eta 0:00:00a 0:00:01


In [16]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(
    data=str(YOUR_YOLO/"data.yaml"),
    epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH, patience=30,
    project=str(WORK/"runs"), name="base8_640",
    device=[0,1],
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15, shear=10, scale=0.5,
    fliplr=0.5, flipud=0.5, mosaic=1.0, copy_paste=0.3,
    lr0=0.01, seed=SEED,
)
BEST = str(WORK/"runs/base8_640/weights/best.pt")
print("best ->", BEST)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo8/data.yaml, degrees=15, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr

In [17]:
res = model.val(data=str(YOUR_YOLO/"data.yaml"), split="val")
print(f"\n=== 8-CLASS BASELINE (640px, with negatives) ===")
print(f"mAP@0.5      {res.box.map50:.4f}")
print(f"mAP@0.5:0.95 {res.box.map:.4f}")
print(f"precision    {res.box.mp:.4f}")
print(f"recall       {res.box.mr:.4f}")
print("\nper-class mAP@0.5:")
for i,n in res.names.items():
    m=res.box.maps[i] if i<len(res.box.maps) else 0
    print(f"  {n:<22} {m:.3f}")

Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,415,896 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3498.1±788.4 MB/s, size: 258.0 KB)
val: Scanning /kaggle/working/yolo8/labels/val.cache... 125 images, 24 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 125/125 40.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 4.4it/s 1.8s0.2s
                   all        125        107      0.799      0.743      0.757      0.524
      Component Liftup          4          4      0.774          1      0.845      0.583
     Component Missing          9          9      0.896      0.956      0.962      0.728
   Component No Solder         16         17        0.5      0.471      0.424       0.31
  Component Solder Dry         24         27      0.872      0.852      0.796      0.576
        Polarity Wrong          7     

In [18]:
from ultralytics import YOLO

def train_at_res(imgsz, batch, name):
    m = YOLO(MODEL)
    m.train(
        data=str(YOUR_YOLO/"data.yaml"),
        epochs=EPOCHS, imgsz=imgsz, batch=batch, patience=30,
        project=str(WORK/"runs"), name=name,
        device=[0,1],
        # IDENTICAL aug/schedule to baseline — only imgsz + batch change
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=15, shear=10, scale=0.5,
        fliplr=0.5, flipud=0.5, mosaic=1.0, copy_paste=0.3,
        lr0=0.01, seed=SEED,
    )
    r = m.val(data=str(YOUR_YOLO/"data.yaml"), split="val")
    return m, r

# 960px (batch 12 is T4-safe at this res)
model_960, res_960 = train_at_res(960, 12, "res_960")

# 1280px (batch 6-8; drop if OOM)
model_1280, res_1280 = train_at_res(1280, 6, "res_1280")

Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo8/data.yaml, degrees=15, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=res_960, nbs=

In [ ]:
print(f"{'class':<22} {'640':>7} {'960':>7} {'1280':>7}")
print("-"*46)
base_maps = res.box.maps          
for i,n in res.names.items():
    b   = base_maps[i]      if i<len(base_maps)      else 0
    m9  = res_960.box.maps[i]  if i<len(res_960.box.maps)  else 0
    m12 = res_1280.box.maps[i] if i<len(res_1280.box.maps) else 0
    print(f"{n:<22} {b:>7.3f} {m9:>7.3f} {m12:>7.3f}")
print("-"*46)
print(f"{'mAP@0.5 (mean)':<22} {res.box.map50:>7.3f} {res_960.box.map50:>7.3f} {res_1280.box.map50:>7.3f}")
print(f"{'mAP@0.5:0.95':<22} {res.box.map:>7.3f} {res_960.box.map:>7.3f} {res_1280.box.map:>7.3f}")

class                      640     960    1280
----------------------------------------------
Component Liftup         0.583   0.576   0.552
Component Missing        0.728   0.665   0.728
Component No Solder      0.310   0.130   0.260
Component Solder Dry     0.576   0.473   0.585
Polarity Wrong           0.562   0.597   0.303
RYB Wrong Sequence       0.553   0.579   0.404
Solder Ball              0.425   0.411   0.415
Solder Short             0.453   0.452   0.442
----------------------------------------------
mAP@0.5 (mean)           0.757   0.763   0.725
mAP@0.5:0.95             0.524   0.485   0.461


In [8]:
K_FOLDS = 5   # 5-fold cross-validation

def stratified_kfold(coco, k, seed=SEED):
    cls_freq=Counter(a["category_id"] for a in coco["annotations"])
    img_cls=defaultdict(set)
    for a in coco["annotations"]: img_cls[a["image_id"]].add(a["category_id"])
    buckets=defaultdict(list)
    for im in coco["images"]:
        cls=img_cls.get(im["id"],set())
        key=min(cls,key=lambda c:cls_freq[c]) if cls else "__none__"
        buckets[key].append(im["id"])
    rng=random.Random(seed); fold_of={}
    for key,ids in buckets.items():
        ids=ids[:]; rng.shuffle(ids)
        for j,iid in enumerate(ids): fold_of[iid]=j%k
    return fold_of

fold_of = stratified_kfold(coco8, K_FOLDS)
folds=defaultdict(list)
for iid,f in fold_of.items(): folds[f].append(iid)
print("fold sizes:", {f:len(folds[f]) for f in sorted(folds)})

# also split negatives across folds (each fold's val gets a share)
neg_shuffled = neg_files[:]; random.Random(SEED).shuffle(neg_shuffled)
neg_shuffled = neg_shuffled[:N_NEG_TOTAL]
neg_folds = defaultdict(list)
for i,p in enumerate(neg_shuffled): neg_folds[i%K_FOLDS].append(p)
print("negatives per fold:", {f:len(neg_folds[f]) for f in sorted(neg_folds)})

fold sizes: {0: 105, 1: 102, 2: 101, 3: 101, 4: 99}
negatives per fold: {0: 24, 1: 24, 2: 24, 3: 24, 4: 24}


In [9]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.7 MB/s eta 0:00:00a 0:00:01


In [10]:
from ultralytics import YOLO
import yaml, numpy as np

# fewer epochs per fold since we train K times; early stopping handles the rest
CV_EPOCHS = 120

all_img_ids = [im["id"] for im in coco8["images"]]
fold_results = []   # list of (map50, map5095, precision, recall, per_class_dict)

for f in range(K_FOLDS):
    print(f"\n{'='*50}\nFOLD {f+1}/{K_FOLDS}\n{'='*50}")
    val_ids   = set(folds[f])
    train_ids = set(all_img_ids) - val_ids

    # build this fold's YOLO dataset fresh
    fold_root = WORK/f"cv_fold{f}"
    shutil.rmtree(fold_root, ignore_errors=True)
    ysl = lambda n: resolve_your_image(n)
    write_yolo_split(coco8, train_ids, "train", fold_root, ysl, NAMES8)
    write_yolo_split(coco8, val_ids,   "val",   fold_root, ysl, NAMES8)
    # negatives: this fold's share -> val; the rest -> train
    val_negs = neg_folds[f]
    train_negs = [p for ff in range(K_FOLDS) if ff!=f for p in neg_folds[ff]]
    for split,chunk in (("train",train_negs),("val",val_negs)):
        (fold_root/f"images/{split}").mkdir(parents=True,exist_ok=True)
        (fold_root/f"labels/{split}").mkdir(parents=True,exist_ok=True)
        for p in chunk:
            shutil.copy(p, fold_root/f"images/{split}/{p.name}")
            (fold_root/f"labels/{split}/{p.stem}.txt").write_text("")
    (fold_root/"data.yaml").write_text(yaml.safe_dump(
        {"path":str(fold_root),"train":"images/train","val":"images/val",
         "names":{i:n for i,n in enumerate(NAMES8)}},sort_keys=False))

    # train
    m = YOLO(MODEL)
    m.train(data=str(fold_root/"data.yaml"), epochs=CV_EPOCHS, imgsz=IMG_SIZE,
            batch=BATCH, patience=25, project=str(WORK/"runs"), name=f"cv_fold{f}",
            device=[0,1], hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=15, shear=10,
            scale=0.5, fliplr=0.5, flipud=0.5, mosaic=1.0, copy_paste=0.3,
            lr0=0.01, seed=SEED, verbose=False)
    r = m.val(data=str(fold_root/"data.yaml"), split="val", verbose=False)
    per_class = {r.names[i]: (r.box.maps[i] if i<len(r.box.maps) else 0.0)
                 for i in r.names}
    fold_results.append((r.box.map50, r.box.map, r.box.mp, r.box.mr, per_class))
    print(f"fold {f+1}: mAP@0.5={r.box.map50:.4f}  mAP@0.5:0.95={r.box.map:.4f}")

    # free disk between folds (Kaggle space is limited)
    shutil.rmtree(fold_root, ignore_errors=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

FOLD 1/5
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/cv_fold0/data.yaml, degrees=15, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=

In [11]:
import numpy as np

map50s = np.array([r[0] for r in fold_results])
map5095s = np.array([r[1] for r in fold_results])
precs = np.array([r[2] for r in fold_results])
recs  = np.array([r[3] for r in fold_results])

print("="*56)
print(f"{K_FOLDS}-FOLD CROSS-VALIDATION RESULTS (mean ± std)")
print("="*56)
print(f"mAP@0.5       {map50s.mean():.4f} ± {map50s.std():.4f}")
print(f"mAP@0.5:0.95  {map5095s.mean():.4f} ± {map5095s.std():.4f}")
print(f"precision     {precs.mean():.4f} ± {precs.std():.4f}")
print(f"recall        {recs.mean():.4f} ± {recs.std():.4f}")

print("\nper-class mAP@0.5 (mean ± std across folds):")
for n in NAMES8:
    vals = np.array([r[4].get(n, 0.0) for r in fold_results])
    print(f"  {n:<22} {vals.mean():.3f} ± {vals.std():.3f}")

print("\nInterpretation: the ± is your real measurement uncertainty.")
print("Two configs differ meaningfully only if the gap exceeds ~1 std.")

5-FOLD CROSS-VALIDATION RESULTS (mean ± std)
mAP@0.5       0.7614 ± 0.0214
mAP@0.5:0.95  0.4998 ± 0.0201
precision     0.7893 ± 0.0403
recall        0.7181 ± 0.0301

per-class mAP@0.5 (mean ± std across folds):
  Component Liftup       0.468 ± 0.070
  Component Missing      0.564 ± 0.119
  Component No Solder    0.321 ± 0.122
  Component Solder Dry   0.640 ± 0.043
  Polarity Wrong         0.583 ± 0.150
  RYB Wrong Sequence     0.557 ± 0.083
  Solder Ball            0.445 ± 0.097
  Solder Short           0.420 ± 0.054

Interpretation: the ± is your real measurement uncertainty.
Two configs differ meaningfully only if the gap exceeds ~1 std.
